# Characterization with Obsidian

This notebook demonstrates how to use Obsidian for a process characterization
campaign.

**Characterization** is the task of identifying regions of the parameter space
where a response meets robustness requirements. To translate this into
mathematical terms, we want to find all points in the parameter space where the
response is above or below a user-defined threshold $h$ with high confidence.
This is different from optimization, where the goal is to find the single best
point.

## Contents
1. Single-objective characterization
2. Multi-objective characterization (joint evaluation)
3. Sequential characterization for Multi-Objective Setup
4. Evaluation with ground truth

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obsidian.campaign import Campaign
from obsidian.parameters import ParamSpace, Param_Continuous, Target

## 1. Single-Objective Characterization

### Problem Setup

We're characterizing a manufacturing process where:
- Parameters: temperature (100-200°C) and pressure (1-10 bar)
- Response: yield (%)
- **Threshold: yield ≥ 60%**

Goal: Identify all parameter combinations that achieve ≥60% yield.

In [ ]:
# Define parameter space
X_space = ParamSpace([Param_Continuous("temperature", 100, 200), Param_Continuous("pressure", 1, 10)])

# Define target with threshold
target = Target("yield", aim="max", threshold=60.0)

# Create campaign
campaign = Campaign(X_space, target, seed=114514, task="characterization")

print(f"Target: {target.name}")
print(f"Threshold: {target.threshold}")
print(f"Aim: {target.aim}")

### Generate Initial Data

We'll use a simple synthetic function that has a sweet spot where yield is high.

In [ ]:
def true_yield_function(temperature, pressure):
    """Synthetic yield function with a sweet spot around T=150, P=6."""
    # Normalize to [0, 1]
    t_norm = (temperature - 100) / 100
    p_norm = (pressure - 1) / 9
    
    # Peak at center with some noise structure
    yield_val = 100 * np.exp(-((t_norm - 0.5)**2 + (p_norm - 0.55)**2) / 0.15)
    yield_val += 10 * np.sin(3 * t_norm) * np.cos(3 * p_norm)
    
    return np.clip(yield_val, 0, 100)

# Generate initial training data using LHS
X_init = campaign.initialize(m_initial=8, method='LHS')
y_init = pd.DataFrame({
    'yield': [true_yield_function(row['temperature'], row['pressure']) 
              for _, row in X_init.iterrows()]
})

data_init = pd.concat([X_init, y_init], axis=1)
print("Initial training data:")
print(data_init)

### Fit the Model

When we fit the campaign, **characterization metrics are automatically computed** if a threshold is set!

In [ ]:
campaign.add_data(data_init)
campaign.fit()

print("Model fitted!")
print("\nCharacterization metrics automatically computed:")
char_cols = [col for col in campaign.data.columns if 'Characterization' in col]
for col in char_cols:
    value = campaign.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

### Understanding the Metrics

Three confidence levels are reported automatically:

| Metric | Description |
|--------|-------------|
| **Pass % (mean)** | Fraction where the GP mean exceeds the threshold. Pass and fail always sums to 100%. Reflects the model's predicted pass/fail split of the space. |
| **Pass/Fail/Classified % (70% CI)** | Classification requiring 70% confidence. Classified % < 100%. Uncertain regions are left unclassified. |
| **Pass/Fail/Classified % (95% CI)** | More conservative. Classified % is lower. Only regions where the model is highly confident are classified. |

The CI metrics show progress. As more data is collected, Classified % grows
toward 100% and 70% and 95% metrics converge. However, due to the inherent
difficulty of characterization, the size of classified space may remain small
even with more data.

### Iterative Characterization

Use characterization acquisition functions to explore the threshold boundary.
Acquisition functions available for single-objective characterization include:
- Pure statistical methods:
  - "SF": Space filling
  - "RS": Random sampling
- Data-driven methods:
  - "STR": Straddle
  - "RANDSTR": Randomized straddle (default)

Pass `acquisition=["RANDSTR"]` explicitly, or rely on the default which is selected based on the `task` set on the campaign.

In [ ]:
# Suggest new experiments using RANDSTR (randomized straddle) acquisition
X_suggest, eval_suggest = campaign.suggest(m_batch=4, acquisition=["RANDSTR"])

print("Suggested experiments:")
print(X_suggest)

# Simulate experiments
y_suggest = pd.DataFrame(
    {"yield": [true_yield_function(row["temperature"], row["pressure"]) for _, row in X_suggest.iterrows()]}
)

# Add data and refit
data_suggest = pd.concat([X_suggest, y_suggest], axis=1)
campaign.add_data(data_suggest)
campaign.fit()

print("\nModel refitted with new data!")
print("\nUpdated characterization metrics:")
for col in char_cols:
    value = campaign.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

### Visualize the Characterization

Let's evaluate the characterization over a grid to visualize the classified regions.

In [ ]:
# Import the unified characterization mosaic plotter
from obsidian.plotting.mpl import plot_2d_response_map

# Pass/fail mosaic
fig, axes = plot_2d_response_map(
    campaign,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    PI_range=0.7,
)
plt.show()

# Continuous mean-prediction surface (single target only)
fig, axes = plot_2d_response_map(
    campaign,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    mode="continuous",
    PI_range=0.7,
)
plt.show()

# 4-level confidence mosaic (Fail / Uncertain / Likely / Confident)
fig, axes = plot_2d_response_map(
    campaign,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    mode="confidence",
)
plt.show()


## 2. Multi-Objective Characterization

### Problem Setup

Now we have **two requirements** that must be satisfied simultaneously:
- **Yield ≥ 75%** (maximize)
- **Cost ≤ 30** (minimize)

Goal: Find all parameter regions that satisfy **both** requirements.

In [ ]:
# Define parameter space (same as before)
X_space_multi = ParamSpace([Param_Continuous("temperature", 100, 200), Param_Continuous("pressure", 1, 10)])

# Both targets with thresholds
targets_multi = [Target("yield", aim="max", threshold=75.0), Target("cost", aim="min", threshold=30.0)]

# Create multi-objective characterization campaign
campaign_multi = Campaign(X_space_multi, targets_multi, task="characterization", seed=42)

print("Multi-objective characterization:")
for target in targets_multi:
    print(f"  {target.name}: {target.aim}, threshold={target.threshold}")

In [ ]:
# Define cost function (inversely related to yield)
def true_cost_function(temperature, pressure):
    """Cost tends to be lower in the sweet spot but with different tradeoffs."""
    t_norm = (temperature - 100) / 100
    p_norm = (pressure - 1) / 9

    # Cost is lower in a slightly different region
    cost = 40 - 20 * np.exp(-((t_norm - 0.4) ** 2 + (p_norm - 0.6) ** 2) / 0.2)
    cost += 5 * np.sin(4 * t_norm) * np.cos(2 * p_norm)

    return np.clip(cost, 10, 50)


# Generate initial data
X_init_multi = campaign_multi.initialize(m_initial=10, method="LHS")
y_init_multi = pd.DataFrame(
    {
        "yield": [true_yield_function(row["temperature"], row["pressure"]) for _, row in X_init_multi.iterrows()],
        "cost": [true_cost_function(row["temperature"], row["pressure"]) for _, row in X_init_multi.iterrows()],
    }
)

data_init_multi = pd.concat([X_init_multi, y_init_multi], axis=1)

print("Initial training data:")
print(data_init_multi[["temperature", "pressure", "yield", "cost"]])

In [ ]:
# Fit the model
campaign_multi.add_data(data_init_multi)
campaign_multi.fit()

print("\n  Multi-objective model fitted!")
print("\n  Multi-objective characterization metrics:")

# Show per-target metrics
char_cols_multi = sorted([col for col in campaign_multi.data.columns if "Characterization" in col])
for col in char_cols_multi:
    value = campaign_multi.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

### Understanding Joint Metrics

Notice the **"Joint"** metrics:
- **Joint Pass %**: Regions where **ALL** targets pass their thresholds
- **Joint Fail %**: Regions where **AT LEAST ONE** target fails
- **Joint Classified %**: Total percentage that can be confidently classified

This tells us how much of the parameter space satisfies **both** requirements simultaneously.

### Iterative Characterization

Acquisition functions available for multi-objective characterization include:
- Pure statistical methods:
  - "SF": Space filling
  - "RS": Random sampling
- Data-driven methods:
  - "MSTR": Straddle extended to multi-objective
  - "JAREX": Randomized straddle extended to multi-objective (default)

Pass `acquisition=["JAREX"]` explicitly, or rely on the default which is selected based on the `task` set on the campaign.

In [ ]:
# Suggest new experiments using JAREX (joint randomized straddle) acquisition
X_suggest_multi, eval_suggest_multi = campaign_multi.suggest(m_batch=5, acquisition=["JAREX"])

print("Suggested experiments:")
print(X_suggest_multi)

# Simulate experiments
y_suggest_multi = pd.DataFrame(
    {
        "yield": [true_yield_function(row["temperature"], row["pressure"]) for _, row in X_suggest_multi.iterrows()],
        "cost": [true_cost_function(row["temperature"], row["pressure"]) for _, row in X_suggest_multi.iterrows()],
    }
)

data_suggest_multi = pd.concat([X_suggest_multi, y_suggest_multi], axis=1)
campaign_multi.add_data(data_suggest_multi)
campaign_multi.fit()

print("\nModel refitted!")
print("\nUpdated joint metrics:")
joint_cols = [col for col in char_cols_multi if "Joint" in col]
for col in joint_cols:
    value = campaign_multi.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

### Visualize Multi-Objective Characterization

In [ ]:
# Multi-objective pass/fail with a joint panel
fig, axes = plot_2d_response_map(
    campaign_multi,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    PI_range=0.7,
    include_joint=True,
)
plt.show()

# Continuous mode requires exactly one target — pick one explicitly
fig, axes = plot_2d_response_map(
    campaign_multi,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    mode="continuous",
    target_names=[campaign_multi.target[0].name],
    PI_range=0.7,
)
plt.show()

# Confidence mode (multi-target) with joint = elementwise min across targets
fig, axes = plot_2d_response_map(
    campaign_multi,
    row_params={"temperature": [100, 125, 150, 175, 200]},
    col_params={"pressure": [0, 2.5, 5.0, 7.5, 10]},
    mode="confidence",
    include_joint=True,
)
plt.show()


## 3. Sequential Characterization with Multi-Target Setup

While empirically it's often more efficient to characterize multiple objectives
simultaneously, there might be cases where you may want to characterize them
sequentially. For example:
- First characterize yield
- Then characterize cost

You can do this with a multi-objective campaign using either:
1. **Explicit target selection**: Pass specific targets to `suggest()` (preferred)
2. **Masking with `tracking_only`**: Temporarily exclude targets from
   optimization

For option 2 to work, you need to mask all but **one** target with
`tracking_only=True`. While this works numerically, masking a `Target` object
with `tracking_only=True` will also exclude it from automatic characterization
metric calculation, which may not be desirable. Therefore, we recommend using
explicit target selection when characterizing sequentially.

### Approach 1: Explicit Target Selection

Simply pass specific targets to `suggest()` to focus on those targets.

In [ ]:
# Create campaign with both targets active
targets_explicit = [Target("yield", aim="max", threshold=75.0), Target("cost", aim="min", threshold=30.0)]

campaign_explicit = Campaign(X_space_multi, targets_explicit, task="characterization", seed=42)
campaign_explicit.add_data(data_init_multi)
campaign_explicit.fit()

print("Explicit target selection approach")
print("\nPhase 1: Focus on yield by passing target=[yield_target]")

# Suggest for yield only — pass target= to restrict which output the acquisition uses
X_suggest_yield, _ = campaign_explicit.suggest(
    m_batch=3, target=[campaign_explicit.target[0]], acquisition=["RANDSTR"]  # yield only
)

print("Yield-focused suggestions:")
print(X_suggest_yield)

y_suggest_explicit = pd.DataFrame(
    {
        "yield": [true_yield_function(row["temperature"], row["pressure"]) for _, row in X_suggest_yield.iterrows()],
        "cost": [true_cost_function(row["temperature"], row["pressure"]) for _, row in X_suggest_yield.iterrows()],
    }
)

campaign_explicit.add_data(pd.concat([X_suggest_yield, y_suggest_explicit], axis=1))
campaign_explicit.fit()

print("\nYield phase complete!")
print("\nPhase 2: Focus on cost by passing target=[cost_target]")

X_suggest_cost, _ = campaign_explicit.suggest(
    m_batch=3, target=[campaign_explicit.target[1]], acquisition=["RANDSTR"]  # cost only
)

print("Cost-focused suggestions:")
print(X_suggest_cost)

### Approach 2: Using `tracking_only`

Alternatively, mark targets as `tracking_only=True` to exclude them from suggestions while still tracking their values.

In [ ]:
# Create campaign with cost as tracking-only initially
targets_sequential = [
    Target('yield', aim='max', threshold=75.0, tracking_only=False),
    Target('cost', aim='min', threshold=30.0, tracking_only=True)  # Track but don't optimize
]

campaign_seq = Campaign(X_space_multi, targets_sequential, task='characterization', seed=42)

# Add initial data
campaign_seq.add_data(data_init_multi)
campaign_seq.fit()

print("Sequential characterization - Phase 1: Yield only")
print("\nTarget configuration:")
for target in campaign_seq.target:
    print(f"  {target.name}: tracking_only={target.tracking_only}")

char_cols_seq = [col for col in campaign_seq.data.columns if 'Characterization' in col and 'yield' in col.lower()]
print("\nYield characterization metrics:")
for col in char_cols_seq:
    value = campaign_seq.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

In [ ]:
# Suggest focusing on yield characterization
X_suggest_seq1, _ = campaign_seq.suggest(
    m_batch=4,
    acquisition=['RANDSTR']  # Single-objective since only yield is active
)

print("Phase 1 suggestions (yield characterization):")
print(X_suggest_seq1)

# Simulate and add data
y_suggest_seq1 = pd.DataFrame({
    'yield': [true_yield_function(row['temperature'], row['pressure']) 
              for _, row in X_suggest_seq1.iterrows()],
    'cost': [true_cost_function(row['temperature'], row['pressure']) 
             for _, row in X_suggest_seq1.iterrows()]
})

campaign_seq.add_data(pd.concat([X_suggest_seq1, y_suggest_seq1], axis=1))
campaign_seq.fit()

print("\nPhase 1 complete!")

In [ ]:
# Switch to cost characterization
campaign_seq.target[0].tracking_only = True   # track yield only
campaign_seq.target[1].tracking_only = False  # activate cost

print("Sequential characterization - Phase 2: Cost only")
print("\nUpdated target configuration:")
for target in campaign_seq.target:
    print(f"  {target.name}: tracking_only={target.tracking_only}")

# Refit to update characterization metrics (now only cost is active)
campaign_seq.fit()

char_cols_cost = [col for col in campaign_seq.data.columns if 'Characterization' in col and 'cost' in col.lower()]
print("\nCost characterization metrics:")
for col in char_cols_cost:
    value = campaign_seq.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

In [ ]:
campaign_seq.data.columns

In [ ]:
campaign_seq.data

In [ ]:
# Suggest focusing on cost characterization
X_suggest_seq2, _ = campaign_seq.suggest(m_batch=4, acquisition=["RANDSTR"])

print("Phase 2 suggestions (cost characterization):")
print(X_suggest_seq2)

y_suggest_seq2 = pd.DataFrame(
    {
        "yield": [true_yield_function(row["temperature"], row["pressure"]) for _, row in X_suggest_seq2.iterrows()],
        "cost": [true_cost_function(row["temperature"], row["pressure"]) for _, row in X_suggest_seq2.iterrows()],
    }
)

campaign_seq.add_data(pd.concat([X_suggest_seq2, y_suggest_seq2], axis=1))
campaign_seq.fit()

print("\nPhase 2 complete!")
print(f"\nTotal experiments: {len(campaign_seq.data)}")

In [ ]:
# Show per-target metrics
char_cols_multi = sorted([col for col in campaign_seq.data.columns if "Characterization" in col])
for col in char_cols_multi:
    value = campaign_seq.data[col].iloc[0]
    print(f"  {col}: {value:.1f}%")

The downside is obvious: tracking-only targets are excluded from
characterization metrics, so you won't see their classification progress in the
metrics until you unmask them. Additionally, there is no joint evaluation at
all, which usually is a key reason for characterizing multiple objectives in the
first place. Therefore, we recommend using explicit target selection for
sequential characterization to retain visibility into all targets and their
joint behavior.

## 4. Evaluation with Ground Truth

When you have access to ground truth (e.g., in benchmarking), you can compute accuracy metrics like Jaccard index and confusion matrices.

In [ ]:
# Use our earlier single-objective campaign
# Evaluate with ground truth on a test set
X_test = pd.DataFrame({
    'temperature': np.random.uniform(100, 200, 100),
    'pressure': np.random.uniform(1, 10, 100)
})

y_test_true = np.array([true_yield_function(row['temperature'], row['pressure']) 
                        for _, row in X_test.iterrows()])

# Evaluate with ground truth
gt_results = campaign.score_against_ground_truth(
    X=X_test,
    y_true=y_test_true,
    PI_range=0.7
)

print("Ground truth evaluation results:")
print(f"\nTarget: {next(iter(gt_results.keys()))}")
print(f"Jaccard index: {next(iter(gt_results.values()))['jaccard']:.3f}")
print("\nConfusion matrix:")
cm = next(iter(gt_results.values()))['confusion_matrix']
print(f"  True Positives (TP):  {cm['TP']:3d}")
print(f"  True Negatives (TN):  {cm['TN']:3d}")
print(f"  False Positives (FP): {cm['FP']:3d}")
print(f"  False Negatives (FN): {cm['FN']:3d}")

# Calculate additional metrics
precision = cm['TP'] / (cm['TP'] + cm['FP']) if (cm['TP'] + cm['FP']) > 0 else 0
recall = cm['TP'] / (cm['TP'] + cm['FN']) if (cm['TP'] + cm['FN']) > 0 else 0
accuracy = (cm['TP'] + cm['TN']) / (cm['TP'] + cm['TN'] + cm['FP'] + cm['FN'])

print(f"\nAdditional metrics:")
print(f"  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  Accuracy:  {accuracy:.3f}")

## 5. Method Comparison: Adaptive vs DOE vs Space-Filling

How do different experiment-selection strategies compare for characterization?
The 2-parameter examples above are smooth and easy — almost any design that
reaches into the interior does well there, so they don't discriminate between
methods. To make the comparison meaningful we switch to a **harder problem**:

- **4 parameters** (`x1..x4`, each on `[0, 1]`) instead of 2.
- A **multi-modal** response: the pass region is a union of several disconnected
  "islands" (a mixture of Gaussian bumps), not one convex blob.
- Thresholds chosen so the pass region is a **small fraction** of the space
  (~35% single-objective, ~8% joint), so the threshold boundary is genuinely
  hard to locate.

This is the regime where *where you sample* matters. We compare three strategies,
scoring each with the ground-truth **Jaccard index** from the previous section:

- **Adaptive** — RANDSTR (single-objective) / JAREX (multi-objective), run
  iteratively up to a fixed budget.
- **Space-filling (SF)** — an iterative baseline that maximizes the minimum
  distance to existing points (`suggest(acquisition=["SF"])`), same loop and
  budget as the adaptive run.
- **DOE** — a one-shot full 2-level factorial (`method="DOE_full"`), fit once at
  its natural size (`2**d + 3` = 19 runs for `d=4`).

The iterative runs start from just a few initial LHS points so you can see the
whole learning curve. Everything below reuses the same public API demonstrated
earlier — only the test problem is new.

In [ ]:
# --- Test problems: multi-modal responses with a small pass region ---
# The pass region is a union of Gaussian "islands", so the threshold boundary is
# nonconvex and occupies only a small fraction of the (high-dimensional) space.

def _bump_mixture(U, centers, weights, length):
    """Sum of isotropic Gaussian bumps evaluated at rows of U (shape (n, d))."""
    U = np.atleast_2d(U)
    val = np.zeros(len(U))
    for c, w in zip(centers, weights):
        d2 = np.sum((U - c) ** 2, axis=1)
        val += w * np.exp(-d2 / (2 * length ** 2))
    return val


# ---- 4-D problem: yield + cost ----
_Y4_CENTERS = np.array([[0.30, 0.30, 0.70, 0.50],
                        [0.72, 0.62, 0.28, 0.40],
                        [0.50, 0.82, 0.58, 0.72]])
_Y4_WEIGHTS = np.array([1.00, 0.90, 0.85])
_C4_CENTERS = np.array([[0.55, 0.35, 0.45, 0.60],
                        [0.25, 0.70, 0.65, 0.30]])
_C4_WEIGHTS = np.array([1.00, 0.90])


def _yield_4d(U):
    """Multi-modal yield surface in [0, 100]; pass region is yield >= threshold."""
    return np.clip(100 * _bump_mixture(U, _Y4_CENTERS, _Y4_WEIGHTS, 0.30), 0, 100)


def _cost_4d(U):
    """Multi-modal cost surface in [5, 50]; pass region is cost <= threshold."""
    return np.clip(50 - 35 * _bump_mixture(U, _C4_CENTERS, _C4_WEIGHTS, 0.32), 5, 50)


# ---- 10-D problem: yield only ----
# Bump centers drawn once from a fixed seed so the surface is reproducible; wider
# bumps (larger length scale) keep the pass region learnable in 10 dimensions.
_Y10_CENTERS = np.random.default_rng(7).uniform(0.25, 0.75, size=(3, 10))
_Y10_WEIGHTS = np.array([1.00, 0.90, 0.85])


def _yield_10d(U):
    """Multi-modal 10-D yield surface in [0, 100]."""
    return np.clip(100 * _bump_mixture(U, _Y10_CENTERS, _Y10_WEIGHTS, 0.45), 0, 100)


def make_problem(names, funcs):
    """Bundle a comparison problem: parameter names, ParamSpace, and response funcs."""
    space = ParamSpace([Param_Continuous(n, 0.0, 1.0) for n in names])
    return {"names": names, "space": space, "funcs": funcs}


PROB_4D = make_problem(["x1", "x2", "x3", "x4"], {"yield": _yield_4d, "cost": _cost_4d})
PROB_10D = make_problem([f"x{i+1}" for i in range(10)], {"yield": _yield_10d})


# --- Shared helpers for the method comparison ---
# Thin wrappers around the same public API used above
# (Campaign.initialize / suggest / fit / score_against_ground_truth).
# `prob` defaults to the 4-D problem; pass PROB_10D for the high-dimensional case.

def make_test_points(prob=PROB_4D, n=4000, seed=0):
    """Fixed random test set in `prob`'s space; every method is scored on it."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame(rng.random((n, len(prob["names"]))), columns=prob["names"])


def true_y(X, targets, prob=PROB_4D):
    """Ground-truth responses for `X`, shape (n_points, n_targets)."""
    U = X[prob["names"]].values
    return np.array([prob["funcs"][t.name](U) for t in targets]).T


def _simulate(X, targets, prob):
    """Build a response DataFrame for suggested points X."""
    U = X[prob["names"]].values
    return pd.DataFrame({t.name: prob["funcs"][t.name](U) for t in targets})


def _jaccard(campaign, targets, X_test, y_test, PI_range=0.7):
    """Per-target jaccard (single objective) or the joint jaccard (multi objective)."""
    res = campaign.score_against_ground_truth(X=X_test, y_true=y_test, PI_range=PI_range)
    key = "Joint" if len(targets) > 1 else targets[0].name
    return res[key]["jaccard"]


def run_adaptive(targets, acq, budget, m_init, m_batch, seed, X_test, y_test, prob=PROB_4D):
    """Iterative loop (works for adaptive RANDSTR/JAREX and the SF baseline).

    Returns (campaign, learning_curve) where learning_curve is an array of
    (n_experiments, jaccard) recorded after each fit.
    """
    # Fresh targets per run so state never leaks between methods
    targets = [Target(t.name, aim=t.aim, threshold=t.threshold) for t in targets]
    campaign = Campaign(prob["space"], targets, seed=seed, task="characterization")

    X_init = campaign.initialize(m_initial=m_init, method="LHS")
    campaign.add_data(pd.concat([X_init, _simulate(X_init, targets, prob)], axis=1))
    campaign.fit()

    curve = [(len(campaign.data), _jaccard(campaign, targets, X_test, y_test))]
    while len(campaign.data) < budget:
        step = min(m_batch, budget - len(campaign.data))
        result = campaign.suggest(m_batch=step, acquisition=[acq])
        assert result is not None, f"suggest() failed for acquisition={acq!r}"
        X_new, _ = result
        campaign.add_data(pd.concat([X_new, _simulate(X_new, targets, prob)], axis=1))
        campaign.fit()
        curve.append((len(campaign.data), _jaccard(campaign, targets, X_test, y_test)))

    return campaign, np.array(curve)


def run_doe(targets, seed, X_test, y_test, prob=PROB_4D, method="DOE_full"):
    """One-shot factorial design, fit once at its NATURAL size (never padded).

    - ``method="DOE_full"``: full 2-level factorial + 3 center points
      (``2**d + 3`` runs; 19 for d=4).
    - ``method="DOE_res4"``: an efficient resolution-IV+ *fractional* factorial,
      far fewer runs in high dimensions (a full factorial would be ``2**10`` in
      the 10-D case). The designer chooses the run count.

    Returns (campaign, (n_experiments, jaccard)).
    """
    targets = [Target(t.name, aim=t.aim, threshold=t.threshold) for t in targets]
    campaign = Campaign(prob["space"], targets, seed=seed, task="characterization")

    if method == "DOE_full":
        n_doe = 2 ** len(prob["names"]) + 3  # 2**d corners + 3 center points
        X_doe = campaign.initialize(m_initial=n_doe, method=method)
    else:
        X_doe = campaign.initialize(method=method)  # fractional design picks its size
    campaign.add_data(pd.concat([X_doe, _simulate(X_doe, targets, prob)], axis=1))
    campaign.fit()
    return campaign, (len(campaign.data), _jaccard(campaign, targets, X_test, y_test))


def plot_comparison(curves, doe_point, title, doe_label="DOE (full factorial"):
    """Plot iterative learning curves plus the one-shot DOE reference point."""
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, curve in curves.items():
        ax.plot(curve[:, 0], curve[:, 1], marker="o", label=label)
    ax.scatter([doe_point[0]], [doe_point[1]], marker="*", s=220, color="black",
               zorder=5, label=f"{doe_label}, {int(doe_point[0])} runs)")
    ax.set_xlabel("Number of experiments")
    ax.set_ylabel("Jaccard index (vs ground truth)")
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig, ax

### Single-Objective Comparison

We characterize the region where **yield ≥ 50** (about 35% of the 4-D space,
spread across several disconnected islands) and track the **Jaccard index**
(overlap between the predicted and true pass regions) as experiments accumulate:

- **RANDSTR** — adaptive characterization (iterative).
- **SF** — space-filling, `suggest(acquisition=["SF"])` (iterative, same loop).
- **DOE** — one-shot full 2-level factorial (19 runs), fit once.

The adaptive and space-filling runs share the same initial LHS design and seed,
so any difference comes purely from *where each method chooses to sample next*.

In [ ]:
# Single-objective comparison on the 4-D yield >= 50 problem
targets_sobj = [Target("yield", aim="max", threshold=50.0)]
X_test = make_test_points(n=4000)
y_test_s = true_y(X_test, targets_sobj)  # shape (n_points, 1)
print(f"True pass fraction (yield >= 50): {(y_test_s[:, 0] >= 50).mean():.1%}")

BUDGET, M_INIT, M_BATCH, SEED_S = 63, 3, 5, 114514

# RANDSTR (adaptive) and SF (space-filling) share the same loop, budget and seed
_, curve_randstr = run_adaptive(targets_sobj, "RANDSTR", BUDGET, M_INIT, M_BATCH, SEED_S, X_test, y_test_s)
_, curve_sf_s    = run_adaptive(targets_sobj, "SF",      BUDGET, M_INIT, M_BATCH, SEED_S, X_test, y_test_s)
# DOE: one-shot full 2-level factorial at its natural size (2**4 + 3 = 19 runs)
_, doe_point_s = run_doe(targets_sobj, SEED_S, X_test, y_test_s)

print(f"\nFinal Jaccard @ {BUDGET} experiments (DOE at {int(doe_point_s[0])} runs):")
print(f"  RANDSTR (adaptive):     {curve_randstr[-1, 1]:.3f}")
print(f"  SF (space-filling):     {curve_sf_s[-1, 1]:.3f}")
print(f"  DOE (full factorial):   {doe_point_s[1]:.3f}")

plot_comparison(
    {"RANDSTR (adaptive)": curve_randstr, "SF (space-filling)": curve_sf_s},
    doe_point_s,
    "Single-objective characterization (4-D, multi-modal): Jaccard vs experiments",
)
plt.show()

### Multi-Objective (Joint) Comparison

Harder still: find where **both** yield ≥ 50 *and* cost ≤ 22 hold simultaneously.
The two responses peak in *different* islands, so the joint pass region is only
~8% of the space. We score every method with the **joint** Jaccard index (the
`"Joint"` entry from `score_against_ground_truth`), which measures overlap
between predicted-joint-pass and truly-joint-pass regions.

In [ ]:
# Multi-objective comparison on the joint (yield >= 50 AND cost <= 22) problem
targets_mobj = [
    Target("yield", aim="max", threshold=50.0),
    Target("cost", aim="min", threshold=22.0),
]
y_test_m = true_y(X_test, targets_mobj)  # shape (n_points, 2)
joint_true = (y_test_m[:, 0] >= 50) & (y_test_m[:, 1] <= 22)
print(f"True JOINT pass fraction: {joint_true.mean():.1%}")

SEED_M = 42

# JAREX (adaptive) and SF (space-filling) share the same loop, budget and seed
_, curve_jarex = run_adaptive(targets_mobj, "JAREX", BUDGET, M_INIT, M_BATCH, SEED_M, X_test, y_test_m)
_, curve_sf_m  = run_adaptive(targets_mobj, "SF",    BUDGET, M_INIT, M_BATCH, SEED_M, X_test, y_test_m)
# DOE: one-shot full 2-level factorial at its natural size (2**4 + 3 = 19 runs)
_, doe_point_m = run_doe(targets_mobj, SEED_M, X_test, y_test_m)

print(f"\nFinal JOINT Jaccard @ {BUDGET} experiments (DOE at {int(doe_point_m[0])} runs):")
print(f"  JAREX (adaptive):       {curve_jarex[-1, 1]:.3f}")
print(f"  SF (space-filling):     {curve_sf_m[-1, 1]:.3f}")
print(f"  DOE (full factorial):   {doe_point_m[1]:.3f}")

plot_comparison(
    {"JAREX (adaptive)": curve_jarex, "SF (space-filling)": curve_sf_m},
    doe_point_m,
    "Multi-objective (joint, 4-D) characterization: Jaccard vs experiments",
)
plt.show()

### Scaling Up: 10-D Single-Objective Comparison

The advantage of boundary-seeking sampling grows with dimensionality. Here we
repeat the single-objective comparison on a **10-parameter** multi-modal problem
(`yield ≥ 40`, roughly a quarter of the space) — the same helpers, just
`prob=PROB_10D`.

Two things change at higher dimension:

- The iterative methods need a **larger budget** (the boundary is a thin shell in
  10-D), so we run to 100 experiments.
- A *full* 2-level factorial would be `2**10 = 1024` runs — absurd as a "cheap"
  one-shot design. The realistic factorial choice is a **fractional
  resolution-IV+** design (`method="DOE_res4"`), which the designer sizes
  automatically (~67 runs).

In [ ]:
# Single-objective comparison on the 10-D yield >= 40 problem
targets_10d = [Target("yield", aim="max", threshold=40.0)]
X_test_10d = make_test_points(PROB_10D, n=4000)
y_test_10d = true_y(X_test_10d, targets_10d, PROB_10D)  # shape (n_points, 1)
print(f"True pass fraction (10-D, yield >= 40): {(y_test_10d[:, 0] >= 40).mean():.1%}")

BUDGET_10D, M_INIT_10D, M_BATCH_10D = 100, 5, 8

# RANDSTR (adaptive) and SF (space-filling) share the same loop, budget and seed
_, curve_randstr_10d = run_adaptive(
    targets_10d, "RANDSTR", BUDGET_10D, M_INIT_10D, M_BATCH_10D, SEED_S, X_test_10d, y_test_10d, PROB_10D
)
_, curve_sf_10d = run_adaptive(
    targets_10d, "SF", BUDGET_10D, M_INIT_10D, M_BATCH_10D, SEED_S, X_test_10d, y_test_10d, PROB_10D
)
# DOE: fractional resolution-IV+ design (a full factorial would be 2**10 = 1024 runs)
_, doe_point_10d = run_doe(targets_10d, SEED_S, X_test_10d, y_test_10d, PROB_10D, method="DOE_res4")

print(f"\nFinal Jaccard @ {BUDGET_10D} experiments (DOE at {int(doe_point_10d[0])} runs):")
print(f"  RANDSTR (adaptive):        {curve_randstr_10d[-1, 1]:.3f}")
print(f"  SF (space-filling):        {curve_sf_10d[-1, 1]:.3f}")
print(f"  DOE (fractional Res IV+):  {doe_point_10d[1]:.3f}")

plot_comparison(
    {"RANDSTR (adaptive)": curve_randstr_10d, "SF (space-filling)": curve_sf_10d},
    doe_point_10d,
    "Single-objective characterization (10-D, multi-modal): Jaccard vs experiments",
    doe_label="DOE (fractional Res IV+",
)
plt.show()

### Takeaway

Across the 4-D and 10-D problems the strategies clearly separate, and the gap
widens with dimensionality:

- **Adaptive (RANDSTR / JAREX) reaches the highest Jaccard fastest.** Once it has
  a rough model, it places new experiments right on the predicted threshold
  boundary — the only region where new data changes the pass/fail verdict. With a
  small pass region scattered across several islands, that focus pays off; the
  advantage is larger for the joint (multi-objective) case where the feasible
  region is smaller still, and larger again at 10-D.

- **Space-filling (SF) improves steadily but lags.** It keeps spreading points
  evenly regardless of the responses, so much of its budget lands in regions
  already confidently classified. It is a solid, model-light baseline on the 4-D
  problem, but by 10-D even coverage barely dents the thin boundary shell within
  budget — adaptive pulls far ahead.

- **The one-shot DOE stays near zero.** A 2-level factorial places every point at
  a corner or the center of the box, far from the interior islands where the pass
  region lives. In 4-D that is 19 corner points; in 10-D a full factorial would
  be `2**10 = 1024` runs, so even the efficient fractional Res IV+ design (~67
  runs) classifies almost nothing. Factorials are excellent for *screening* main
  effects and interactions — they are simply the wrong tool for mapping a
  nonlinear, multi-modal pass/fail boundary.

The general lesson: characterization rewards experiments placed *in the interior,
near the boundary*. Adaptive acquisitions target that boundary directly;
space-filling reaches the interior but spends effort everywhere; corner-based
factorial designs never get there at all. The gap widens with dimensionality,
with narrower/multi-modal pass regions, and with tighter budgets.

## Summary

This notebook demonstrated:

1. **Single-Objective Characterization**
   - Set `task='characterization'` and a `threshold` on the Target
   - Automatic computation of characterization metrics on every `fit()`
   - Use RANDSTR acquisition for efficient boundary exploration

2. **Multi-Objective Characterization**
   - Set thresholds on multiple targets
   - Joint classification metrics (Pass ALL vs Fail ANY)
   - Use JAREX acquisition for multi-objective boundary exploration

3. **Sequential Characterization**
   - Two approaches: `tracking_only` masking or explicit target selection via `suggest(target=...)`
   - Allows focused characterization of individual targets sequentially

4. **Ground Truth Evaluation**
   - Jaccard index and confusion matrix for benchmarking
   - Available via `campaign.score_against_ground_truth()`

5. **Method Comparison: Adaptive vs DOE vs Space-Filling**
   - Benchmarked on harder 4-D and 10-D multi-modal problems where sampling strategy matters
   - Adaptive (RANDSTR / JAREX) reaches the highest Jaccard fastest; space-filling (`SF`) lags and falls further behind at 10-D; a one-shot factorial (`DOE_full` / fractional `DOE_res4`) barely classifies anything

### Key Takeaways

- **Specify `task='characterization'` explicitly** — do not rely on threshold-based inference
- **Metrics are automatic** — classification columns appear in `campaign.data` after every `fit()`
- **Three reporting levels** — mean (always 100% classified), 70% CI, 95% CI
- **Joint metrics for multi-target** — understand where ALL requirements are met simultaneously
- **Flexible workflows** — joint or sequential characterization as needed
- **Characterization wants interior, near-boundary points** — adaptive acquisitions target the boundary directly and win on hard problems, increasingly so as dimensionality grows; space-filling is a reasonable baseline; corner-based factorial designs are best kept for screening